In [1]:
import pandas as pd

data=pd.read_csv('../data/processed/cleaned_dataset.csv')

In [3]:
score = (
    5*data["injuries_fatal"]
    + 3*data["injuries_incapacitating"]
    + 1*data["injuries_non_incapacitating"]
    + 0.5*data["injuries_reported_not_evident"]
)
data["severity_score"] = score

summary = data.groupby("most_severe_injury")["severity_score"].describe()[["count","mean","50%","75%","max"]]
summary.sort_values("mean", ascending=False)

,count,mean,50%,75%,max
most_severe_injury,,,,,
FATAL,351.0,7.202279,5.5,8.0,36.5
INCAPACITATING INJURY,6563.0,3.981639,3.0,4.0,25.0
NONINCAPACITATING INJURY,31521.0,1.436804,1.0,2.0,21.0
"REPORTED, NOT EVIDENT",16073.0,0.680054,0.5,1.0,7.5
NO INDICATION OF INJURY,154767.0,0.000000,0.0,0.0,0.0


In [4]:
data.head()

,crash_date,traffic_control_device,weather_condition,lighting_condition,first_crash_type,trafficway_type,alignment,roadway_surface_cond,road_defect,crash_type,...,injuries_reported_not_evident,injuries_no_indication,crash_hour,crash_day_of_week,crash_month,crash_datetime,crash_year,crash_day_name,any_injury,severity_score
0,07/29/2023 01:00:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,TURNING,NOT DIVIDED,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,0.0,3.0,13,7,7,2023-07-29 13:00:00,2023,Saturday,0,0.0
1,08/13/2023 12:11:00 AM,TRAFFIC SIGNAL,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,0.0,2.0,0,1,8,2023-08-13 00:11:00,2023,Sunday,0,0.0
2,12/09/2021 10:30:00 AM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,0.0,3.0,10,5,12,2021-12-09 10:30:00,2021,Thursday,0,0.0
3,08/09/2023 07:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,ANGLE,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,...,0.0,0.0,19,4,8,2023-08-09 19:55:00,2023,Wednesday,1,5.0
4,08/19/2023 02:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,0.0,3.0,14,7,8,2023-08-19 14:55:00,2023,Saturday,0,0.0


In [6]:
data['severity_score']

0         0.0
1         0.0
2         0.0
3         5.0
4         0.0
         ... 
209270    0.0
209271    0.0
209272    6.0
209273    1.0
209274    0.0
Name: severity_score, Length: 209275, dtype: float64

In [7]:
data["severity_score"] = (
    5*data["injuries_fatal"]
    + 3*data["injuries_incapacitating"]
    + 1*data["injuries_non_incapacitating"]
    + 0.5*data["injuries_reported_not_evident"]
)

thr = data["severity_score"].quantile(0.90)
data["y_severe_score"] = (data["severity_score"] >= thr).astype(int)

data["y_severe_score"].mean(), thr

(np.float64(0.2028813761796679), np.float64(1.0))

In [9]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score

data = data.copy().sample(n=20000, random_state=42)

data["severity_score"] = (
    5*data["injuries_fatal"]
    + 3*data["injuries_incapacitating"]
    + 1*data["injuries_non_incapacitating"]
    + 0.5*data["injuries_reported_not_evident"]
)

exclude_cols = {
    "severity_score",
    "crash_type",
    "most_severe_injury",
    "any_injury",
    "injuries_total",
    "injuries_fatal",
    "injuries_incapacitating",
    "injuries_non_incapacitating",
    "injuries_reported_not_evident",
    "injuries_no_indication",
    "injuries_unknown",
    "injuries_possible",
    "injuries_minor",
    "injuries_major",
}

X = data[[c for c in data.columns if c not in exclude_cols]]
y = data["severity_score"]

cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=20,
)

pipe = Pipeline([("pre", pre), ("rf", rf)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 4))
print("R2 :", round(r2_score(y_test, pred), 4))

perm = permutation_importance(
    pipe, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

imp = pd.DataFrame(
    {"feature": X.columns, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std}
).sort_values("importance_mean", ascending=False)

imp.head(20)

MAE: 0.259
R2 : 0.5535


,feature,importance_mean,importance_std
19,y_severe_score,9.998302e-01,1.897025e-02
12,num_units,1.262789e-02,3.782706e-03
10,damage,8.641057e-03,3.279945e-03
4,first_crash_type,7.142450e-03,1.759413e-03
13,crash_hour,3.353036e-03,3.453538e-03
7,roadway_surface_cond,1.564678e-03,1.686499e-03
15,crash_month,1.483480e-03,1.638805e-03
17,crash_year,1.446483e-03,9.980072e-04
1,traffic_control_device,1.056203e-03,7.510185e-04
18,crash_day_name,7.771389e-04,3.993740e-04


The permutation-importance results from the random-forest model suggest that crash context variables—especially num_units, damage, first_crash_type, and crash_hour—are the strongest predictors of the severity score. In contrast, factors such as weather_condition, road_defect, and several geometry/intersection indicators contribute little to the model, indicating that the severity score is driven primarily by the impact characteristics and exposure/traffic complexity rather than weather-related conditions in this dataset.